# Assignment 6 — CNN II (Regression)

This notebook implements **two regression tasks** using convolutional neural networks:

1. **Clock Reading** — given an image of a clock face, predict the time in minutes
2. **Triangle Counting** — given an image with 0–20 triangles, predict the count

### Why Regression (not Classification)?
Unlike classification (predicting a *category*), regression predicts a *continuous number*.
Time in minutes (0–719) and triangle count (0–20) are both continuous-ish quantities, so
we use **MSE / circular MSE loss** instead of cross-entropy.

### Key outputs
- Training + validation loss curves (so we can spot overfitting / non-convergence)
- Visual prediction grids (predicted clock face vs. true clock face)
- Error distribution histograms

## 0. Imports & Setup

We use:
- **PyTorch** — deep learning framework (model, loss, optimiser)
- **PIL / ImageDraw** — to synthetically *generate* our datasets
- **matplotlib** — visualisation

In [ ]:
# Uncomment the line below if running on Google Colab:
# %pip install -q torch torchvision pillow matplotlib

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import random
import math
from pathlib import Path
import shutil

from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Use GPU if available, otherwise fall back to CPU.
# Training on CPU is slower but produces identical results.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
# Setting seeds ensures the same random numbers are drawn every run,
# making results reproducible. We set seeds for Python's random module,
# NumPy, and PyTorch.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE      = 96    # Each image is 96×96 pixels (greyscale)
BATCH         = 64    # Number of images processed per gradient update
EPOCHS_CLOCK  = 40    # Clock gets more epochs — harder task
EPOCHS_TRI    = 30    # Triangle counting converges faster
#
# Why these values?
#   IMG_SIZE=96  — small enough to be fast, large enough to see clock hands
#   BATCH=64     — common default; larger = faster but needs more RAM

# ── Directory layout ─────────────────────────────────────────────────────────
ROOT        = Path('data_assign6')
CLOCK_RAW   = ROOT / 'clock_raw'
CLOCK_SPLIT = ROOT / 'clock_split'
TRI_RAW     = ROOT / 'tri_raw'
TRI_SPLIT   = ROOT / 'tri_split'

for p in [CLOCK_RAW, CLOCK_SPLIT, TRI_RAW, TRI_SPLIT]:
    p.mkdir(parents=True, exist_ok=True)

print('Directories ready.')

---
## Part 1 — Clock Regression

### Task
Given a greyscale image of an analogue clock, predict the time as **total minutes** (0–719).

### Why is this hard?
- Time is **circular**: 11:59 and 12:00 are 1 minute apart, but naively their numeric
  encodings (719 and 0) are far apart.
- The dataset is **small** (720 images = 12 hours × 60 minutes), so the model must
  generalise from limited examples.

### Solution — sin/cos encoding

The classic approach encodes time as a scalar in [0, 1] with a sigmoid output.
The problem is that **sigmoid physically cannot represent circularity** — 0.0 and 1.0
are at opposite ends of the output range, but represent times 1 minute apart.

Instead, we encode the target as **two values**:
```
fraction = total_minutes / 720
sin_target = sin(2π × fraction)
cos_target = cos(2π × fraction)
```

This maps time onto the unit circle in 2D. Now 11:59 and 12:00 are naturally
adjacent points, and standard MSE loss works perfectly — no special circular loss
needed. At inference we recover minutes via `atan2(sin, cos)`.

### Data augmentation
720 images is tiny; adding **Gaussian noise augmentation** on the fly effectively
multiplies our training set and helps the model generalise.

In [ ]:
# ── Clock image generator ────────────────────────────────────────────────────
# We synthesise our own dataset because real clock images are hard to label.
# Each image is greyscale (mode='L'), white background, with two hands drawn
# using trigonometry:
#
#   angle = 2π × (fraction of full circle) − π/2
#               └─ −π/2 rotates 0° to the 12 o'clock position
#
def render_clock(h, m, size=96, noise_std=0.0):
    """
    Draw a simple analogue clock showing hour h, minute m.
    noise_std > 0 adds Gaussian pixel noise (data augmentation).
    """
    img  = Image.new('L', (size, size), 255)   # white background
    draw = ImageDraw.Draw(img)
    c    = size // 2                            # centre pixel
    r    = int(size * 0.4)                      # clock radius

    # Outer circle (the clock face)
    draw.ellipse((c-r, c-r, c+r, c+r), outline=0)

    # Minute hand angle: one full rotation per 60 minutes
    ang_m = 2 * math.pi * (m / 60) - math.pi / 2
    # Hour hand angle: one full rotation per 12 hours; also moves with minutes
    ang_h = 2 * math.pi * ((h % 12 + m / 60) / 12) - math.pi / 2

    # Minute hand — longer (0.8r), thinner (width=2)
    mx = c + int(r * 0.8 * math.cos(ang_m))
    my = c + int(r * 0.8 * math.sin(ang_m))
    draw.line((c, c, mx, my), fill=0, width=2)

    # Hour hand — shorter (0.5r), thicker (width=3)
    hx = c + int(r * 0.5 * math.cos(ang_h))
    hy = c + int(r * 0.5 * math.sin(ang_h))
    draw.line((c, c, hx, hy), fill=0, width=3)

    # Optional: add tiny tick marks at each hour position
    for tick in range(12):
        ang_t = 2 * math.pi * (tick / 12) - math.pi / 2
        x0 = c + int(r * 0.88 * math.cos(ang_t))
        y0 = c + int(r * 0.88 * math.sin(ang_t))
        x1 = c + int(r * 1.00 * math.cos(ang_t))
        y1 = c + int(r * 1.00 * math.sin(ang_t))
        draw.line((x0, y0, x1, y1), fill=0, width=1)

    # Apply pixel noise for data augmentation
    if noise_std > 0:
        arr  = np.array(img, dtype=np.float32)
        arr += np.random.normal(0, noise_std * 255, arr.shape)
        arr  = np.clip(arr, 0, 255).astype(np.uint8)
        img  = Image.fromarray(arr)

    return img


# Generate the dataset: 12 hours × 60 minutes = 720 images
if len(list(CLOCK_RAW.glob('*.png'))) != 720:
    print('Generating clock images...')
    for h in range(12):
        hh = 12 if h == 0 else h
        for m in range(60):
            img = render_clock(hh, m, IMG_SIZE)
            img.save(CLOCK_RAW / f'clock_{hh:02d}_{m:02d}.png')

print(f'Clock images in dataset: {len(list(CLOCK_RAW.glob("*.png")))}')

# Quick sanity check: show a few clocks with their labels
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
sample_times = [(12,0),(3,15),(6,30),(9,45),(1,5),(11,59)]
for ax, (h, m) in zip(axes, sample_times):
    ax.imshow(render_clock(h, m, IMG_SIZE), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{h:02d}:{m:02d}', fontsize=10)
    ax.axis('off')
plt.suptitle('Sample clock images', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Train / Val / Test split ──────────────────────────────────────────────────
# We divide the dataset into three non-overlapping subsets:
#
#   Train (70%) — the model sees these images and updates its weights on them
#   Val   (15%) — used to *monitor* learning; weights are NOT updated on val
#   Test  (15%) — held out until the very end; gives an unbiased final score
#
# The key rule: never let the model train on validation or test data!

def split_dataset(src, dst):
    """Randomly split images from src into train/val/test folders under dst."""
    files = list(src.glob('*.png'))
    random.shuffle(files)
    n  = len(files)
    tr = files[:int(0.70 * n)]
    va = files[int(0.70 * n):int(0.85 * n)]
    te = files[int(0.85 * n):]

    for split_name in ['train', 'val', 'test']:
        p = dst / split_name
        if p.exists():
            shutil.rmtree(p)   # remove old split to avoid stale files
        p.mkdir(parents=True)

    for f in tr: shutil.copy(f, dst / 'train' / f.name)
    for f in va: shutil.copy(f, dst / 'val'   / f.name)
    for f in te: shutil.copy(f, dst / 'test'  / f.name)

    print(f'  {src.name}: {len(tr)} train | {len(va)} val | {len(te)} test')

split_dataset(CLOCK_RAW, CLOCK_SPLIT)

In [ ]:
# ── Dataset class ─────────────────────────────────────────────────────────────
# PyTorch's Dataset abstraction requires two methods:
#   __len__     — returns the total number of samples
#   __getitem__ — returns one (image, label) pair by index
#
# Labels (targets):
#   minutes = (hour % 12) * 60 + minute   → range 0–719
#   encoded as (sin, cos) to handle circularity:
#     fraction = minutes / 720
#     sin_target = sin(2π × fraction)
#     cos_target = cos(2π × fraction)
#
# Why sin/cos instead of a scalar?
#   With a scalar in [0,1] + sigmoid, the model sees 0.0 and 1.0 as
#   maximally far apart, but they represent 12:00 and 11:59 — only
#   1 minute difference! Sin/cos maps time onto a circle, so adjacent
#   times are always close together in output space.

class ClockDS(Dataset):
    def __init__(self, path, augment=False):
        """
        path    — folder containing PNG clock images
        augment — if True, add Gaussian noise to each image on the fly
                  (only used for training, not val/test)
        """
        self.augment = augment
        self.files   = list(path.glob('*.png'))
        self.X = []
        self.y = []

        for f in self.files:
            # Load as numpy array, scale pixels to [0, 1]
            img = np.array(Image.open(f)) / 255.0
            # Add channel dimension: (H, W) → (1, H, W)
            # CNNs expect (C, H, W) — C=1 for greyscale
            self.X.append(img[None, :, :])

            # Parse label from filename, e.g. 'clock_03_45.png' → h=3, m=45
            parts = f.stem.split('_')   # ['clock', '03', '45']
            h, m  = int(parts[1]), int(parts[2])
            minutes = (h % 12) * 60 + m

            # Encode as (sin, cos) on the unit circle
            frac  = minutes / 720.0
            angle = 2 * math.pi * frac
            self.y.append([math.sin(angle), math.cos(angle)])

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = self.X[i]
        if self.augment:
            # Add small random noise: makes the model more robust
            # (acts as a cheap form of regularisation for our tiny dataset)
            x = x + torch.randn_like(x) * 0.05
            x = x.clamp(0, 1)   # keep pixels in valid range
        return x, self.y[i]


# augment=True on training set; never on val/test (we want clean evaluation)
train_ds = ClockDS(CLOCK_SPLIT / 'train', augment=True)
val_ds   = ClockDS(CLOCK_SPLIT / 'val',   augment=False)
test_ds  = ClockDS(CLOCK_SPLIT / 'test',  augment=False)

# DataLoader wraps the Dataset, handles batching and shuffling
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH)
test_dl  = DataLoader(test_ds,  batch_size=BATCH)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')
print(f'Batch shape example: {next(iter(train_dl))[0].shape}')   # expect [64, 1, 96, 96]
print(f'Target shape example: {next(iter(train_dl))[1].shape}')  # expect [64, 2]

In [ ]:
# ── CNN Architecture ──────────────────────────────────────────────────────────
#
# Architecture overview:
#
#   Input  (1, 96, 96)
#     ↓  Conv2d(1→16)  + BN + ReLU + MaxPool  →  (16, 48, 48)
#     ↓  Conv2d(16→32) + BN + ReLU + MaxPool  →  (32, 24, 24)
#     ↓  Conv2d(32→64) + BN + ReLU + MaxPool  →  (64, 12, 12)
#     ↓  Conv2d(64→64) + BN + ReLU + AdaptiveAvgPool(4,4) → (64, 4, 4)
#     ↓  Flatten → 1024 features
#     ↓  Linear(1024→64) + ReLU + Dropout
#     ↓  Linear(64→2) → (sin, cos) — NO sigmoid!
#
# Key design choices:
#   Conv2d — learns spatial filters (detects edges, hands, circles)
#   MaxPool — downsamples; makes the model translation-invariant
#   BatchNorm — normalises activations; speeds up training, stabilises gradients
#   AdaptiveAvgPool(4) — keeps some spatial info (hand position matters!)
#   No sigmoid — sin/cos targets range from -1 to +1, sigmoid clips to (0,1)
#   4 conv blocks — extra depth helps distinguish subtle angular differences

class ClockNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            # BatchNorm2d normalises activations — speeds up training and stabilises gradients.
            # This is one of the most impactful additions for small datasets.

            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),

            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4)   # output is (batch, 64, 4, 4)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),            # (batch, 64, 4, 4) → (batch, 1024)
            nn.Linear(1024, 64),
            nn.ReLU(),
            nn.Dropout(0.3),         # randomly zeroes 30% of neurons — reduces overfitting
            nn.Linear(64, 2),        # output: (sin, cos) — raw values, NO activation
        )

    def forward(self, x):
        return self.fc(self.conv(x))   # shape: (batch, 2)


clock_model = ClockNet().to(DEVICE)

# Count trainable parameters
n_params = sum(p.numel() for p in clock_model.parameters() if p.requires_grad)
print(f'ClockNet parameters: {n_params:,}')
print(clock_model)

In [ ]:
# ── Loss & Optimiser ──────────────────────────────────────────────────────────
#
# With sin/cos encoding, we don't need a special circular loss at all!
# Standard MSE on the (sin, cos) targets handles circularity automatically:
#   - 11:59 → (sin≈-0.009, cos≈1.000)
#   - 12:00 → (sin= 0.000, cos= 1.000)
#   - MSE between them ≈ 0.00008 — tiny, as it should be!
#
# Compare this to the old scalar approach where 11:59→0.999 and 12:00→0.000
# gave MSE ≈ 0.998 — nearly maximum loss for a 1-minute difference.

clock_loss_fn = nn.MSELoss()

# Adam optimiser adapts the learning rate per-parameter.
# lr=1e-3 is a good default starting point.
# weight_decay adds L2 regularisation — helps prevent overfitting.
opt = torch.optim.Adam(clock_model.parameters(), lr=1e-3, weight_decay=1e-4)

# ReduceLROnPlateau: if val loss stops improving for 5 epochs, halve the lr.
# This helps the model fine-tune in the later stages of training.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.5, patience=5
)

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
#
# Each epoch:
#   1. TRAIN PHASE — iterate over training batches, compute loss, backprop, update weights
#   2. VAL PHASE   — iterate over val batches with gradients disabled (faster, no memory leak)
#   3. Record both losses so we can plot them later
#
# Gradient steps only happen in the TRAIN phase.

clock_train_losses = []   # track training loss per epoch
clock_val_losses   = []   # track validation loss per epoch

for epoch in range(EPOCHS_CLOCK):
    # ── Train ─────────────────────────────────────────────────────────────────
    clock_model.train()   # switches on Dropout & BatchNorm train behaviour
    running_loss = 0.0

    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)

        opt.zero_grad()                    # 1. clear gradients from last step
        pred = clock_model(x)              # 2. forward pass → (batch, 2)
        loss = clock_loss_fn(pred, y)      # 3. MSE on (sin, cos) targets
        loss.backward()                    # 4. backprop: compute gradients
        opt.step()                         # 5. update weights
        running_loss += loss.item()

    train_loss = running_loss / len(train_dl)

    # ── Validate ───────────────────────────────────────────────────────────────
    clock_model.eval()    # switches off Dropout & puts BatchNorm in eval mode
    val_running = 0.0

    with torch.no_grad():  # no gradient computation → saves memory & time
        for x, y in val_dl:
            x, y  = x.to(DEVICE), y.to(DEVICE)
            pred  = clock_model(x)
            val_running += clock_loss_fn(pred, y).item()

    val_loss = val_running / len(val_dl)

    clock_train_losses.append(train_loss)
    clock_val_losses.append(val_loss)

    # Step scheduler: reduce LR if val_loss is plateauing
    scheduler.step(val_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:02d}/{EPOCHS_CLOCK}  '
              f'train_loss={train_loss:.5f}  val_loss={val_loss:.5f}  '
              f'lr={opt.param_groups[0]["lr"]:.2e}')

In [ ]:
# ── Loss curve ────────────────────────────────────────────────────────────────
#
# Reading the loss curve:
#
#   Both curves fall together → model is learning well
#   Train falls but val stays flat → OVERFITTING (model memorised training data)
#   Both curves stay high → UNDERFITTING (model hasn't learned enough)
#   Val < Train → unusual but can happen with small val sets or strong dropout

fig, ax = plt.subplots(figsize=(8, 4))
epochs = range(1, EPOCHS_CLOCK + 1)
ax.plot(epochs, clock_train_losses, 'b-o', markersize=3, label='Train loss')
ax.plot(epochs, clock_val_losses,   'r-o', markersize=3, label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss (sin/cos targets)')
ax.set_title('Clock model — training vs validation loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Final train loss:', round(clock_train_losses[-1], 5))
print('Final val   loss:', round(clock_val_losses[-1], 5))

gap = clock_val_losses[-1] - clock_train_losses[-1]
if gap > 0.01:
    print(f'⚠️  Gap = {gap:.4f} — signs of overfitting (val > train). '
          'Try more augmentation or more Dropout.')
elif clock_val_losses[-1] > 0.05:
    print('⚠️  Val loss still high — model may need more epochs or a lower LR.')
else:
    print('✅ Training looks healthy!')

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
#
# We measure error in *minutes* so it's human-readable.
# To convert sin/cos predictions back to minutes:
#   angle = atan2(sin, cos)          → range [-π, π]
#   if angle < 0: angle += 2π       → range [0, 2π]
#   minutes = angle / (2π) × 720    → range [0, 720)

def sincos_to_minutes(sin_val, cos_val, total=720):
    """Convert (sin, cos) prediction back to total minutes."""
    angle = math.atan2(sin_val, cos_val)
    if angle < 0:
        angle += 2 * math.pi
    return (angle / (2 * math.pi)) * total


def circ_err_minutes(true_min, pred_min, total=720):
    """Circular error in minutes (handles wrap-around)."""
    d = abs(true_min - pred_min)
    return min(d, total - d)


clock_model.eval()
all_true_min, all_pred_min = [], []
errors_min = []

with torch.no_grad():
    for x, y in test_dl:
        x = x.to(DEVICE)
        pred = clock_model(x).cpu()
        for i in range(len(y)):
            true_m = sincos_to_minutes(y[i][0].item(), y[i][1].item())
            pred_m = sincos_to_minutes(pred[i][0].item(), pred[i][1].item())
            all_true_min.append(true_m)
            all_pred_min.append(pred_m)
            errors_min.append(circ_err_minutes(true_m, pred_m))

mean_err   = np.mean(errors_min)
median_err = np.median(errors_min)
print(f'Clock test — Mean error:   {mean_err:.1f} min')
print(f'Clock test — Median error: {median_err:.1f} min')
print(f'(Random baseline would be ~180 min)')

In [ ]:
# ── Error distribution histogram ──────────────────────────────────────────────
#
# Ideally most errors are near 0. A heavy right tail means some clocks
# confuse the model badly (e.g. ambiguous hand positions).

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(errors_min, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(mean_err,   color='red',    linestyle='--', label=f'Mean {mean_err:.1f} min')
axes[0].axvline(median_err, color='orange', linestyle='--', label=f'Median {median_err:.1f} min')
axes[0].set_xlabel('Error (minutes)')
axes[0].set_ylabel('Count')
axes[0].set_title('Clock error distribution (test set)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: true vs predicted (already in minutes)
axes[1].scatter(all_true_min, all_pred_min, alpha=0.4, s=10, color='steelblue')
axes[1].plot([0, 720], [0, 720], 'r--', linewidth=1, label='Perfect prediction')
axes[1].set_xlabel('True time (minutes)')
axes[1].set_ylabel('Predicted time (minutes)')
axes[1].set_title('True vs Predicted time (minutes)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Visual predictions: predicted vs actual clock ─────────────────────────────
#
# The clearest way to understand model errors: draw the clock the model
# *thinks* it sees (red hands) alongside the true clock (black hands).

def minutes_to_hm(total_min):
    """Convert total minutes (0–719) to (hour, minute)."""
    total_min = int(round(total_min)) % 720
    h = total_min // 60
    m = total_min % 60
    return (12 if h == 0 else h), m


def render_clock_comparison(true_min, pred_min, size=96):
    """
    Draw a single RGB image showing:
      Black hands = true time
      Red   hands = model's prediction
    """
    img  = Image.new('RGB', (size, size), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    c = size // 2
    r = int(size * 0.4)

    draw.ellipse((c-r, c-r, c+r, c+r), outline=(0, 0, 0))

    # Tick marks
    for tick in range(12):
        ang = 2*math.pi*(tick/12) - math.pi/2
        draw.line(
            (c + int(r*0.88*math.cos(ang)), c + int(r*0.88*math.sin(ang)),
             c + int(r*1.00*math.cos(ang)), c + int(r*1.00*math.sin(ang))),
            fill=(0,0,0), width=1
        )

    def draw_hands(total_m, colour, width_m=2, width_h=3):
        h, m = minutes_to_hm(total_m)
        ang_m = 2*math.pi*(m/60) - math.pi/2
        ang_h = 2*math.pi*((h%12 + m/60)/12) - math.pi/2
        draw.line((c, c,
                   c+int(r*0.8*math.cos(ang_m)),
                   c+int(r*0.8*math.sin(ang_m))), fill=colour, width=width_m)
        draw.line((c, c,
                   c+int(r*0.5*math.cos(ang_h)),
                   c+int(r*0.5*math.sin(ang_h))), fill=colour, width=width_h)

    draw_hands(true_min, colour=(0, 0, 0))       # true  → black
    draw_hands(pred_min, colour=(220, 30, 30))    # pred  → red
    return img


# Sort by error so we can show best → worst
# (all_true_min and all_pred_min are already in minutes from the evaluation cell)
sorted_results = sorted(
    zip(errors_min, all_true_min, all_pred_min),
    key=lambda x: x[0]
)

# Show best 4, a couple from the middle, and worst 4
best_4   = sorted_results[:4]
middle_4 = sorted_results[len(sorted_results)//2 - 2 : len(sorted_results)//2 + 2]
worst_4  = sorted_results[-4:]

groups = [('Best predictions', best_4),
          ('Middle predictions', middle_4),
          ('Worst predictions', worst_4)]

fig, big_axes = plt.subplots(3, 4, figsize=(12, 9))

for row_idx, (title, group) in enumerate(groups):
    for col_idx, (err, t_min, p_min) in enumerate(group):
        h_t, m_t = minutes_to_hm(t_min)
        h_p, m_p = minutes_to_hm(p_min)

        comp_img = render_clock_comparison(t_min, p_min, size=IMG_SIZE)
        ax = big_axes[row_idx][col_idx]
        ax.imshow(comp_img)
        ax.set_title(
            f'True: {h_t:02d}:{m_t:02d}\nPred: {h_p:02d}:{m_p:02d}\nErr: {err:.0f}m',
            fontsize=8, color='black'
        )
        ax.axis('off')
    # Row label
    big_axes[row_idx][0].set_ylabel(title, fontsize=10, rotation=90, labelpad=40)

# Legend patches
black_patch = mpatches.Patch(color='black',      label='True time')
red_patch   = mpatches.Patch(color='firebrick',  label='Predicted time')
fig.legend(handles=[black_patch, red_patch], loc='lower center',
           ncol=2, fontsize=10, frameon=True, bbox_to_anchor=(0.5, 0.0))
fig.suptitle('Clock predictions: black=true, red=predicted', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 🔍 Diagnosis — what we changed and why

| Root cause | Symptom | Fix applied |
|---|---|---|
| Scalar [0,1] target + Sigmoid | Can't handle circularity; 11:59 and 12:00 look maximally far apart | **Sin/cos encoding** — maps time onto unit circle; MSE works naturally |
| No loss tracking | Can't see what's happening | Added `train_losses` / `val_losses` lists + plot |
| Tiny dataset (720 imgs) | Overfits quickly | Added noise augmentation on train set |
| Fixed learning rate | Gets stuck in shallow plateaus | Added `ReduceLROnPlateau` scheduler |
| No batch norm | Unstable gradients | Added `BatchNorm2d` after each conv |
| No regularisation | Overfits train set | Added `Dropout(0.3)` in FC layers + `weight_decay` |
| Only 3 conv layers | Insufficient capacity for angular discrimination | Added 4th conv block |
| AdaptiveAvgPool(1) | Throws away all spatial info | Changed to `AdaptiveAvgPool(4)` — keeps some position info |

---
## Part 2 — Triangle Counting

### Task
Given a greyscale image containing 0–20 randomly drawn triangles, predict the **count**.

### Why is this different from clock regression?
- The target is a **count** (integer), not a circular quantity, so plain MSE works fine.
- The model needs to learn to count occurrences of a shape, not read a position.
- We normalise count to [0, 1] the same way (divide by 20).

In [ ]:
# ── Triangle image generator ──────────────────────────────────────────────────
#
# Each image contains n random triangles drawn as outlines (d.polygon).
# We generate 40 images per count class → 21 × 40 = 840 images total.

def tri_img(n, size=96):
    """Draw n random triangles on a white background."""
    img  = Image.new('L', (size, size), 255)
    draw = ImageDraw.Draw(img)
    for _ in range(n):
        # 3 random vertices
        pts = [(random.randint(5, size-5), random.randint(5, size-5))
               for _ in range(3)]
        draw.polygon(pts, outline=0)   # outline only (not filled)
    return img


# Generate dataset: counts 0–20, 40 images each
if len(list(TRI_RAW.glob('*.png'))) == 0:
    print('Generating triangle images...')
    for count in range(21):
        for i in range(40):
            tri_img(count, IMG_SIZE).save(TRI_RAW / f'tri_{count}_{i:02d}.png')

print(f'Triangle images in dataset: {len(list(TRI_RAW.glob("*.png")))}')

# Sanity check: visualise
fig, axes = plt.subplots(1, 7, figsize=(14, 2.5))
for ax, n in zip(axes, [0, 1, 3, 5, 10, 15, 20]):
    ax.imshow(tri_img(n, IMG_SIZE), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'n={n}', fontsize=10)
    ax.axis('off')
plt.suptitle('Sample triangle images', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Split triangle dataset ────────────────────────────────────────────────────
split_dataset(TRI_RAW, TRI_SPLIT)

In [ ]:
# ── Triangle Dataset class ────────────────────────────────────────────────────
#
# Very similar to ClockDS. Label is parsed from the filename:
#   'tri_7_03.png' → count = 7, normalised to 7/20 = 0.35

class TriDS(Dataset):
    def __init__(self, path, augment=False):
        self.augment = augment
        self.files   = list(path.glob('*.png'))
        self.X, self.y = [], []

        for f in self.files:
            img = np.array(Image.open(f)) / 255.0
            self.X.append(img[None, :, :])           # add channel dim
            count = int(f.stem.split('_')[1])
            self.y.append(count / 20.0)              # normalise to [0, 1]

        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = self.X[i]
        if self.augment:
            x = (x + torch.randn_like(x) * 0.04).clamp(0, 1)
        return x, self.y[i]


tr_tr = TriDS(TRI_SPLIT / 'train', augment=True)
tr_va = TriDS(TRI_SPLIT / 'val',   augment=False)
tr_te = TriDS(TRI_SPLIT / 'test',  augment=False)

tr_tr_dl = DataLoader(tr_tr, batch_size=BATCH, shuffle=True)
tr_va_dl = DataLoader(tr_va, batch_size=BATCH)
tr_te_dl = DataLoader(tr_te, batch_size=BATCH)

print(f'Train: {len(tr_tr)}  Val: {len(tr_va)}  Test: {len(tr_te)}')

In [ ]:
# ── Triangle model & training ─────────────────────────────────────────────────
#
# We define a separate TriNet — similar architecture to ClockNet, but with
# 1 output + Sigmoid (since triangle count normalised to [0,1] is NOT circular).
# This time loss = plain MSE.

class TriNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)   # output is (batch, 64, 1, 1)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()          # maps to (0, 1) — appropriate for normalised count
        )

    def forward(self, x):
        return self.fc(self.conv(x)).squeeze(1)


tri_model = TriNet().to(DEVICE)
tri_opt   = torch.optim.Adam(tri_model.parameters(), lr=1e-3)
tri_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    tri_opt, mode='min', factor=0.5, patience=5
)
mse_loss = nn.MSELoss()

tri_n_params = sum(p.numel() for p in tri_model.parameters() if p.requires_grad)
print(f'TriNet parameters: {tri_n_params:,}')

tri_train_losses = []
tri_val_losses   = []

for epoch in range(EPOCHS_TRI):
    # Train
    tri_model.train()
    running = 0.0
    for x, y in tr_tr_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        tri_opt.zero_grad()
        l = mse_loss(tri_model(x), y)
        l.backward()
        tri_opt.step()
        running += l.item()
    train_loss = running / len(tr_tr_dl)

    # Validate
    tri_model.eval()
    val_running = 0.0
    with torch.no_grad():
        for x, y in tr_va_dl:
            val_running += mse_loss(tri_model(x.to(DEVICE)), y.to(DEVICE)).item()
    val_loss = val_running / len(tr_va_dl)

    tri_train_losses.append(train_loss)
    tri_val_losses.append(val_loss)
    tri_sched.step(val_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:02d}/{EPOCHS_TRI}  '
              f'train_loss={train_loss:.5f}  val_loss={val_loss:.5f}')

In [ ]:
# ── Triangle loss curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS_TRI+1), tri_train_losses, 'b-o', markersize=3, label='Train loss')
ax.plot(range(1, EPOCHS_TRI+1), tri_val_losses,   'r-o', markersize=3, label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Triangle model — training vs validation loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Triangle evaluation ───────────────────────────────────────────────────────
#
# Error is reported in triangle counts (not normalised) — much easier to interpret.
# e.g. mean error of 1.5 means on average we're off by 1.5 triangles.

tri_model.eval()
tri_true, tri_pred = [], []
tri_errors = []

with torch.no_grad():
    for x, y in tr_te_dl:
        p = tri_model(x.to(DEVICE)).cpu()
        for yi, pi in zip(y.tolist(), p.tolist()):
            true_count = yi * 20
            pred_count = pi * 20
            tri_true.append(true_count)
            tri_pred.append(pred_count)
            tri_errors.append(abs(true_count - pred_count))

print(f'Triangle test — Mean error:   {np.mean(tri_errors):.2f} triangles')
print(f'Triangle test — Median error: {np.median(tri_errors):.2f} triangles')
print(f'(Random baseline would be ~6.7 triangles)')

In [ ]:
# ── Triangle visual predictions ───────────────────────────────────────────────
#
# Show the actual test images alongside their true and predicted counts.
# This helps identify *which* counts the model struggles with.

# Pair each test file with its error
test_files = list(TRI_SPLIT.glob('test/*.png'))
file_results = []
tri_model.eval()

with torch.no_grad():
    for f in test_files:
        img_arr = np.array(Image.open(f)) / 255.0
        x = torch.tensor(img_arr[None, None, :, :], dtype=torch.float32).to(DEVICE)
        pred_count = tri_model(x).item() * 20
        true_count = int(f.stem.split('_')[1])
        err = abs(true_count - pred_count)
        file_results.append((err, true_count, pred_count, f))

file_results.sort(key=lambda x: x[0])

# Show best 4 and worst 4
samples = file_results[:4] + file_results[-4:]
labels  = ['Best']*4 + ['Worst']*4

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ((err, tc, pc, f), label) in enumerate(zip(samples, labels)):
    ax  = axes[idx // 4][idx % 4]
    img = Image.open(f)
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    col = 'green' if err < 1 else 'red'
    ax.set_title(
        f'{label}\nTrue: {tc}   Pred: {pc:.1f}\nErr: {err:.1f}',
        fontsize=8, color=col
    )
    ax.axis('off')

plt.suptitle('Triangle predictions — best (top row) vs worst (bottom row)', fontsize=12)
plt.tight_layout()
plt.show()

# Scatter: true count vs predicted count
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(tri_true, tri_pred, alpha=0.4, s=15, color='steelblue')
ax.plot([0, 20], [0, 20], 'r--', linewidth=1, label='Perfect')
ax.set_xlabel('True count')
ax.set_ylabel('Predicted count')
ax.set_title('Triangle: True vs Predicted count')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary & Key Takeaways

| Task | Mean error | Random baseline |
|---|---|---|
| Clock reading | *see output above* min | ~180 min |
| Triangle counting | *see output above* triangles | ~6.7 triangles |

### Architecture Choices — Clock

We used a 4-layer CNN with batch normalisation outputting **(sin, cos)** targets:

| Choice | Reason |
|---|---|
| **Sin/cos encoding** | Eliminates circular discontinuity — the single most important fix. Without it, the model can't learn that 11:59 ≈ 12:00. |
| **4 conv blocks** | Extra depth helps discriminate subtle angular differences between clock hands |
| **BatchNorm** | Stabilises training on our tiny 504-image training set |
| **AdaptiveAvgPool(4)** | Retains spatial information — hand *position* matters for reading time |
| **Dropout(0.3) + weight decay** | Regularisation to prevent overfitting on small data |
| **No sigmoid on output** | Sin/cos range from -1 to +1; sigmoid would clip to (0,1) and lose half the circle |

### Architecture Choices — Triangle

Same CNN backbone but simpler — 3 conv blocks, 1 output + Sigmoid:

| Choice | Reason |
|---|---|
| **3 conv blocks** | Counting is simpler than reading angles; less capacity needed |
| **Sigmoid output** | Count normalised to [0,1] is not circular, so sigmoid is appropriate |
| **MSE loss** | Standard regression loss works fine for linear targets |

### If you want to improve further
- **More data** — generate 5× more clock/triangle images
- **Deeper architecture** — add a 5th conv block for clock
- **Horizontal/vertical flips** — more augmentation for triangles
- **Transfer learning** — use a pretrained ResNet backbone (even for greyscale)